# SignSage: dual-model ISL training

Train isolated-sign recognition and continuous pose-to-English translation as separate experiments. This notebook is self-contained, uses PyTorch, and exports backend-shaped research bundles. No real training or downloads start unless an enable flag is changed.

Run cells top to bottom. First code cell is configuration only. Attach datasets through Kaggle **Add data** or set local paths; do not attach raw data to output bundles. Models trained here are experimental and are not promoted or avatar-signing systems.

## Pipelines

1. Isolated signs: video or prepared `(T, 543, 3)` pose sequence → one dataset-specific word label.
2. Continuous translation: aligned `.pose`/pose sequence → English target sentence.

Never combine vocabularies across datasets. iSign text is sentence translation data, not an isolated-sign class map.

## Dataset evidence and attachment policy

Metadata verified from official dataset cards, Hugging Face repositories, and Mendeley/DataCite DOI records. Datasets must be attached manually via Kaggle **Add data** or mapped to local paths; this notebook never performs automated multi-gigabyte downloads.

| Dataset | Verified Metadata & License | Role in Dual Pipeline | 543-Landmark Preprocessing & Conversion Requirement |
|---|---|---|---|
| [GloSL-ISL](https://huggingface.co/datasets/pedroodb/glosl-isl) | Public HF repository; 126,846 rows; `annotations.csv` (`id, split, text, signer, video_ref, pose_file`); CC BY-NC 4.0; null signer metadata. | Primary Model B (Translation) reader. Research-only opt-in. | Native 543-point MediaPipe Holistic `.pose` files. Attach uncompressed folder containing `annotations.csv` and `poses/`. |
| [iSign](https://huggingface.co/datasets/Exploration-Lab/iSign) | Gated HF repository; CC BY-NC-SA 4.0; `uid,text` annotations with large multipart pose archives. | Optional Model B (Translation) reader. Requires HF approval. | Native MediaPipe `.pose` files named `<uid>.pose`. Attach extracted pose directory and CSV. UIDs are sample IDs; video grouping is sample-independent unless explicit `video_id` metadata is supplied. |
| [ISL-CSLTR](https://data.mendeley.com/datasets/kcmpdxky7p/1) | Mendeley Data (DOI 10.17632/kcmpdxky7p.1); CC BY 4.0; 700 continuous videos, 100 sentences, 7 signers. | Candidate Model B (Translation) dataset. | Raw video files (.mp4). Requires MediaPipe Holistic extraction to generate (T, 543, 3) landmarks. Supported via built-in `convert_isl_csltr_to_canonical_manifest()`. |
| [CISLR](https://huggingface.co/datasets/Exploration-Lab/CISLR) | Gated HF repository (AFL-3.0); word-level recognition; raw video archives + 1024-D I3D features. | Candidate Model A (Recognition) dataset. | **Important**: Pre-extracted I3D features from CISLR are 1024-D visual features, NOT 543-point skeletal landmarks. To use SignSage's skeletal pipeline, raw videos must be processed via MediaPipe Holistic into (T, 543, 3) format. |
| [Amrita SLR](https://data.mendeley.com/datasets/dtg9s785v8/1) | Mendeley Data (DOI 10.17632/dtg9s785v8/1); CC BY 4.0 (embargoed access); 50 signs, ~16 clips/class, 7 signers. | Candidate Model A (Recognition) dataset. | 1080p 28 FPS raw video files. Requires MediaPipe Holistic landmark extraction to produce 543-point arrays before isolated-word training. |
| [INCLUDE](https://huggingface.co/datasets/manojkumarcs/INCLUDE_Dataset) | Ungated HF repository (~56 GB); 4,292 videos / 263 signs; **no license declaration** found. | **Disabled by policy** until licensing terms are verified. | Do not download or train until copyright terms are officially declared. |

### 543-Landmark Preprocessing Contract

SignSage enforces a unified skeletal representation across all pipelines:
- **Total Landmarks**: 543 3D coordinates `(x, y, z)` ordered strictly as:
  - `POSE_LANDMARKS` [0:33] (33 points)
  - `FACE_LANDMARKS` [33:501] (468 points)
  - `LEFT_HAND_LANDMARKS` [501:522] (21 points)
  - `RIGHT_HAND_LANDMARKS` [522:543] (21 points)
- **Model A (Recognition)**: Slices body pose [0:33] and both hands [501:543] (75 points = 225 features), normalizes by centering on hip midpoint and scaling by shoulder width, resamples to 64 frames.
- **Model B (Translation)**: Normalizes all 543 landmarks (1629 features), resizes to `MAX_FRAMES` (160), and concatenates frame-to-frame velocity deltas in `FrameProjection` (3258 input dim).

### Pilot vs. Full Training Configuration

- `TRAINING_MODE = "pilot"`: Restricts training and evaluation to a deterministic subset (`MAX_SAMPLES_PER_SPLIT = 256`) for rapid verification in Kaggle sessions. Exported artifacts are explicitly tagged `scope: "pilot_subset"`.
- `TRAINING_MODE = "full"`: Sets `MAX_SAMPLES_PER_SPLIT = 0` to train on the entire attached dataset. Exported artifacts are tagged `scope: "full_dataset"`.


In [ ]:
import csv
import hashlib
import itertools
import json
import math
import os
import random
import tempfile
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset, TensorDataset

SEED = 42
BATCH_SIZE = 8
EPOCHS = 3
LEARNING_RATE = 3e-4

# TRAINING MODE CONFIGURATION
# Set TRAINING_MODE = "pilot" for fast smoke checks (bounded by MAX_SAMPLES_PER_SPLIT).
# Set TRAINING_MODE = "full" for complete dataset training across all available rows.
TRAINING_MODE = "pilot"  # "pilot" | "full"
MAX_SAMPLES_PER_SPLIT = 256 if TRAINING_MODE == "pilot" else 0
TRAINING_SCOPE = "pilot_subset" if (TRAINING_MODE == "pilot" and MAX_SAMPLES_PER_SPLIT > 0) else "full_dataset"

MAX_FRAMES = 160
MAX_TOKENS = 96
SP_VOCAB_SIZE = 4000
MODEL_DIM = 128
ATTENTION_HEADS = 4
TRANSFORMER_LAYERS = 2
FEED_FORWARD_DIM = 512

TRAIN_RECOGNITION = False
TRAIN_TRANSLATION = False
RESUME_RECOGNITION = False
RESUME_TRANSLATION = False
RUN_SYNTHETIC_SMOKE = True
CONFIRM_DATASET_RIGHTS = False
ALLOW_RESEARCH_ONLY_DATASETS = False

ISOLATED_MANIFEST = None  # e.g. Path("/kaggle/input/my-signs/manifest.csv")
ISOLATED_DATA_ROOT = Path("/kaggle/input")
ISOLATED_PATH_COLUMN = "path"
ISOLATED_LABEL_COLUMN = "label"
ISOLATED_VIDEO_COLUMN = "video_id"
ISOLATED_SIGNER_COLUMN = "signer"  # set "" when absent
ISOLATED_DATASET_NAME = "local isolated-sign dataset"
ISOLATED_LICENSE = "unverified"

TRANSLATION_MANIFEST = None  # GloSL annotations.csv, iSign uid/text CSV, or canonical CSV.
TRANSLATION_POSE_ROOT = Path("/kaggle/input")
TRANSLATION_FORMAT = "auto"  # auto | glosl | isign | canonical
TRANSLATION_DATASET_NAME = "GloSL-ISL"
TRANSLATION_LICENSE = "CC BY-NC 4.0"

OUTPUT_ROOT = Path("/kaggle/working/outputs") if Path("/kaggle/working").exists() else Path("outputs")
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True

print(f"device={DEVICE}; seed={SEED}; outputs={OUTPUT_ROOT.resolve()}")
print(f"training_mode={TRAINING_MODE}; scope={TRAINING_SCOPE}; max_samples_per_split={MAX_SAMPLES_PER_SPLIT}")
print(f"train_recognition={TRAIN_RECOGNITION}; train_translation={TRAIN_TRANSLATION}")
if (BATCH_SIZE < 1 or EPOCHS < 1 or MAX_SAMPLES_PER_SPLIT < 0
        or MAX_FRAMES < 2 or MAX_TOKENS < 3 or MODEL_DIM % ATTENTION_HEADS):
    raise ValueError("Invalid training config: check batch/epoch counts, sample cap, sequence lengths, and attention dimensions.")


## 2–3. Dataset loading, validation, and splits

Readers accept explicit local paths only. Translation `.pose` layout is checked before positional landmark slicing. Video extraction preserves frame order and uses the SignSage order: pose, face, left hand, right hand.

In [ ]:
TOTAL_LANDMARKS = 543
RAW_FEATURE_DIM = 1629
COMPONENT_ORDER = ["POSE_LANDMARKS", "FACE_LANDMARKS", "LEFT_HAND_LANDMARKS", "RIGHT_HAND_LANDMARKS"]
COMPONENT_RANGES = {
    "POSE_LANDMARKS": [0, 33], "FACE_LANDMARKS": [33, 501],
    "LEFT_HAND_LANDMARKS": [501, 522], "RIGHT_HAND_LANDMARKS": [522, 543],
}


def read_csv_rows(path):
    path = Path(path)
    if not path.is_file():
        raise FileNotFoundError(f"Dataset CSV not found: {path}")
    with path.open(newline="", encoding="utf-8-sig") as stream:
        reader = csv.DictReader(stream)
        rows = list(reader)
        fields = reader.fieldnames or []
    if not rows:
        raise ValueError(f"No data rows in {path}")
    print(f"{path}: {len(rows):,} rows; fields={fields}")
    return rows, fields


def show_input_tree(root, limit=20):
    root = Path(root)
    if not root.exists():
        print(f"Input root unavailable: {root}")
        return
    sample = [p.relative_to(root).as_posix() for p in itertools.islice(
        (p for p in root.rglob("*") if p.is_file()), limit
    )]
    print(f"{root}: sample files={sample}")


def validate_landmarks(sequence, source):
    arr = np.asarray(sequence)
    if arr.ndim != 3 or arr.shape[1:] != (TOTAL_LANDMARKS, 3):
        raise ValueError(f"{source}: expected (T,543,3), received {arr.shape}")
    if len(arr) < 2:
        raise ValueError(f"{source}: need at least two frames")
    if arr.dtype.kind not in "iuf" or not np.isfinite(arr).all():
        raise ValueError(f"{source}: landmarks must be numeric and finite")
    return arr.astype(np.float32, copy=False)


def _read_pose_format(path):
    try:
        from pose_format import Pose
    except ImportError as exc:
        raise ImportError("Reading .pose requires `pip install pose-format`.") from exc
    pose = Pose.read(Path(path).read_bytes())
    names = [component.name for component in pose.header.components]
    points = sum(len(component.points) for component in pose.header.components)
    if names[:len(COMPONENT_ORDER)] != COMPONENT_ORDER or points < TOTAL_LANDMARKS:
        raise ValueError(f"{path}: incompatible pose header; components={names}, points={points}")
    data = np.asarray(np.ma.filled(pose.body.data, 0.0), dtype=np.float32)
    if data.ndim != 4 or data.shape[1] < 1 or data.shape[2] < TOTAL_LANDMARKS or data.shape[3] < 3:
        raise ValueError(f"{path}: unexpected pose tensor shape {data.shape}")
    return validate_landmarks(data[:, 0, :TOTAL_LANDMARKS, :3], path)


def _video_landmarks(path):
    try:
        import cv2
        import mediapipe as mp
    except ImportError as exc:
        raise ImportError("Video extraction requires `pip install mediapipe opencv-python-headless`.") from exc
    cap = cv2.VideoCapture(str(path))
    if not cap.isOpened():
        raise ValueError(f"Cannot open video: {path}")
    sequences = []
    def points_or_zeros(component, count):
        if component is None:
            return np.zeros((count, 3), dtype=np.float32)
        points = np.asarray([[p.x, p.y, p.z] for p in component.landmark], dtype=np.float32)
        if points.shape != (count, 3):
            raise ValueError(f"Unexpected MediaPipe component shape {points.shape}; expected {(count, 3)}")
        return points
    with mp.solutions.holistic.Holistic(
        static_image_mode=False, model_complexity=1,
        enable_segmentation=False, refine_face_landmarks=False,
    ) as holistic:
        try:
            while True:
                ok, frame = cap.read()
                if not ok:
                    break
                result = holistic.process(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
                sequences.append(np.concatenate([
                    points_or_zeros(result.pose_landmarks, 33),
                    points_or_zeros(result.face_landmarks, 468),
                    points_or_zeros(result.left_hand_landmarks, 21),
                    points_or_zeros(result.right_hand_landmarks, 21),
                ], axis=0))
        finally:
            cap.release()
    return validate_landmarks(np.asarray(sequences), path)


def load_landmarks(path, cache_dir=None):
    path = Path(path)
    if not path.is_file():
        raise FileNotFoundError(f"Sample file not found: {path}")
    suffix = path.suffix.lower()
    if suffix == ".pose":
        return _read_pose_format(path)
    if suffix in (".mp4", ".mov", ".avi", ".mkv"):
        if cache_dir is None:
            return _video_landmarks(path)
        cache_dir = Path(cache_dir)
        cache_dir.mkdir(parents=True, exist_ok=True)
        stat = path.stat()
        key = hashlib.sha256(f"{path.resolve()}:{stat.st_size}:{stat.st_mtime_ns}".encode()).hexdigest()
        cached = cache_dir / f"{key}.npy"
        if not cached.exists():
            np.save(cached, _video_landmarks(path))
        return validate_landmarks(np.load(cached, allow_pickle=False), path)
    if suffix == ".npy":
        return validate_landmarks(np.load(path, allow_pickle=False), path)
    if suffix == ".npz":
        with np.load(path, allow_pickle=False) as archive:
            candidates = [k for k in ("landmarks", "pose", "poses") if k in archive]
            if not candidates:
                raise ValueError(f"{path}: .npz must contain landmarks, pose, or poses")
            return validate_landmarks(archive[candidates[0]], path)
    raise ValueError(f"Unsupported sample format {suffix!r}: {path}")


def normalize_pose_sequence(flat):
    arr = np.asarray(flat)
    if arr.ndim != 2 or arr.shape[1] != RAW_FEATURE_DIM or arr.dtype.kind not in "iuf":
        raise ValueError(f"Expected numeric (T,1629) input, received {arr.shape}/{arr.dtype}")
    x = arr.reshape(len(arr), TOTAL_LANDMARKS, 3).astype(np.float32)
    with np.errstate(invalid="ignore"):
        center = np.nanmean(x[:, [23, 24], :], axis=1, keepdims=True)
    bad = ~np.isfinite(center).all(axis=(1, 2))
    if bad.any():
        with np.errstate(invalid="ignore"):
            center[bad] = np.nanmean(x[bad, :33, :], axis=1, keepdims=True)
    x -= center
    with np.errstate(invalid="ignore"):
        scales = np.nanstd(x[:, :33, :].reshape(len(x), -1), axis=1)
    valid = scales[np.isfinite(scales) & (scales > 1e-6)]
    scale = float(np.nanmedian(valid)) if len(valid) else 1.0
    if not math.isfinite(scale) or scale <= 1e-6:
        scale = 1.0
    return np.nan_to_num(x / scale, nan=0.0, posinf=0.0, neginf=0.0).reshape(len(x), -1).astype(np.float32)


def resize_sequence(sequence, target_len):
    seq = np.asarray(sequence)
    if seq.ndim != 2 or not len(seq):
        raise ValueError(f"Expected non-empty (T,D) sequence, got {seq.shape}")
    if len(seq) > target_len:
        idx = np.linspace(0, len(seq) - 1, target_len).round().astype(np.int64)
        return seq[idx].astype(np.float32), target_len
    out = np.zeros((target_len, seq.shape[1]), dtype=np.float32)
    out[:len(seq)] = seq
    return out, len(seq)


def resolve_input_path(value, root):
    candidate = Path(str(value))
    return candidate if candidate.is_absolute() else Path(root) / candidate


def split_grouped(records, group_key, seed=SEED):
    groups = defaultdict(list)
    for row in records:
        group = str(row.get(group_key, "")).strip()
        if not group:
            raise ValueError(f"Missing group value for {group_key}; refusing leaky split.")
        groups[group].append(row)
    if len(groups) < 3:
        raise ValueError(f"Need at least three independent {group_key} groups; found {len(groups)}")
    keys = list(groups)
    random.Random(seed).shuffle(keys)
    targets = {"train": 0.8, "validation": 0.1, "test": 0.1}
    result = {name: [] for name in targets}
    total = sum(len(rows) for rows in groups.values())
    for group in keys:
        name = min(targets, key=lambda key: len(result[key]) / max(targets[key] * total, 1))
        result[name].extend(groups[group])
    all_groups = {name: {r[group_key] for r in rows} for name, rows in result.items()}
    if (all_groups["train"] & all_groups["validation"] or
        all_groups["train"] & all_groups["test"] or
        all_groups["validation"] & all_groups["test"]):
        raise AssertionError(f"Group leakage in {group_key} split")
    return result


def limit_split(rows, limit, seed):
    if not limit or len(rows) <= limit:
        return rows
    return random.Random(seed).sample(rows, limit)


def fingerprint_rows(rows, fields):
    payload = [[str(row.get(field, "")) for field in fields] for row in rows]
    return hashlib.sha256(json.dumps(payload, ensure_ascii=False, separators=(",", ":")).encode()).hexdigest()


def prepare_isolated_features(landmarks):
    arr = validate_landmarks(landmarks, "recognition sample")
    selected = arr[:, list(range(33)) + list(range(501, 543)), :]
    hips = (selected[:, 23] + selected[:, 24]) * 0.5
    shoulder = np.linalg.norm(selected[:, 11, :2] - selected[:, 12, :2], axis=1)
    valid = shoulder[shoulder > 1e-5]
    scale = float(np.median(valid)) if len(valid) else 1.0
    selected = (selected - hips[:, None, :]) / scale
    indices = np.linspace(0, len(selected) - 1, 64).round().astype(int)
    features = selected[indices].reshape(64, 225).astype(np.float32)
    if not np.isfinite(features).all():
        raise ValueError("Non-finite recognition features")
    return features


def load_isolated_records():
    if ISOLATED_MANIFEST is None:
        raise ValueError("Set ISOLATED_MANIFEST to an attached CSV before enabling recognition.")
    rows, fields = read_csv_rows(ISOLATED_MANIFEST)
    required = {ISOLATED_PATH_COLUMN, ISOLATED_LABEL_COLUMN}
    if not required.issubset(fields):
        raise ValueError(f"Recognition manifest missing columns {sorted(required - set(fields))}; found {fields}")
    show_input_tree(ISOLATED_DATA_ROOT)
    records, errors = [], []
    for index, row in enumerate(rows):
        label = str(row.get(ISOLATED_LABEL_COLUMN, "")).strip()
        path = resolve_input_path(row.get(ISOLATED_PATH_COLUMN, ""), ISOLATED_DATA_ROOT)
        try:
            if not label:
                raise ValueError("empty sign label")
            features = prepare_isolated_features(load_landmarks(path, OUTPUT_ROOT / "cache" / "video_poses"))
            video = str(row.get(ISOLATED_VIDEO_COLUMN, "")).strip() or path.stem
            signer = str(row.get(ISOLATED_SIGNER_COLUMN, "")).strip() if ISOLATED_SIGNER_COLUMN else ""
            records.append({"features": features, "label": label, "video_id": video, "signer": signer, "path": str(path)})
        except Exception as exc:
            errors.append(f"row {index + 2} ({path}): {exc}")
    if errors:
        print(f"Invalid recognition samples: {len(errors)}")
        print("\n".join(errors[:20]))
        raise ValueError("Recognition data validation failed; no samples were silently discarded.")
    if len(records) < 3:
        raise ValueError(f"Need at least three valid recognition samples, found {len(records)}")
    labels = sorted({row["label"] for row in records})
    if len(labels) < 2:
        raise ValueError(f"Need at least two isolated-sign classes, found {labels}")
    print(f"recognition rows={len(records)}; classes={len(labels)}; labels={labels}")
    return records


def convert_isl_csltr_to_canonical_manifest(
    video_dir, sentence_mapping_path, output_manifest_path,
    signer_mapping=None
):
    """Convert an attached ISL-CSLTR Mendeley dataset into the canonical translation manifest.

    ISL-CSLTR (10.17632/kcmpdxky7p.1) contains 700 continuous sign videos across 7 signers
    and 100 sentences. This function matches existing video files to sentence targets and
    writes a canonical translation CSV:
        pose_path,text,video_id,split,signer

    Args:
        video_dir: Directory containing ISL-CSLTR video files or extracted landmarks.
        sentence_mapping_path: CSV or text file mapping video/sentence identifiers to English sentences.
        output_manifest_path: Destination path for the canonical CSV.
        signer_mapping: Optional dict mapping video identifiers to signer IDs.
    """
    video_dir = Path(video_dir)
    sentence_mapping_path = Path(sentence_mapping_path)
    output_manifest_path = Path(output_manifest_path)

    if not video_dir.is_dir():
        raise FileNotFoundError(f"ISL-CSLTR video directory not found: {video_dir}")
    if not sentence_mapping_path.is_file():
        raise FileNotFoundError(f"Sentence mapping file not found: {sentence_mapping_path}")

    # Read sentences mapping
    mapping = {}
    with sentence_mapping_path.open(newline="", encoding="utf-8-sig") as stream:
        sample_line = stream.readline()
        stream.seek(0)
        delimiter = "," if "," in sample_line else ("\t" if "\t" in sample_line else None)
        reader = csv.reader(stream, delimiter=delimiter) if delimiter else csv.reader(stream)
        for row in reader:
            if len(row) >= 2:
                key, text = row[0].strip(), row[1].strip()
                if key and text:
                    mapping[key] = text
                    mapping[Path(key).stem] = text

    if not mapping:
        raise ValueError(f"No valid mappings found in {sentence_mapping_path}")

    # Discover video / pose files
    video_files = [p for p in video_dir.rglob("*") if p.is_file() and p.suffix.lower() in (".mp4", ".mov", ".pose", ".npy", ".npz")]
    if not video_files:
        raise FileNotFoundError(f"No video or pose files found in {video_dir}")

    records = []
    missing = []
    for vpath in video_files:
        stem = vpath.stem
        text = mapping.get(stem) or mapping.get(vpath.name)
        if text is None:
            # Check if filename encodes sentence id, e.g. sentence_001
            for k in mapping:
                if k in stem:
                    text = mapping[k]
                    break
        if text:
            # Attempt to extract signer ID from filename or directory structure
            signer = ""
            if signer_mapping and stem in signer_mapping:
                signer = signer_mapping[stem]
            else:
                for part in stem.split("_") + [p.name for p in vpath.parents]:
                    if part.lower().startswith("signer") or (part.lower().startswith("s") and part[1:].isdigit()):
                        signer = part
                        break
            video_id = stem
            records.append({
                "pose_path": str(vpath.resolve()),
                "text": text,
                "video_id": video_id,
                "split": "",  # let assign_translation_splits handle split by signer/video
                "signer": signer,
            })
        else:
            missing.append(str(vpath))

    records.sort(key=lambda r: (r["signer"], r["video_id"]))
    if not records:
        raise ValueError(f"None of the {len(video_files)} video files could be matched to sentences in {sentence_mapping_path}")

    output_manifest_path.parent.mkdir(parents=True, exist_ok=True)
    with output_manifest_path.open("w", newline="", encoding="utf-8") as stream:
        writer = csv.DictWriter(stream, fieldnames=["pose_path", "text", "video_id", "split", "signer"])
        writer.writeheader()
        writer.writerows(records)

    print(f"Generated ISL-CSLTR canonical manifest: {output_manifest_path}")
    print(f"Aligned samples: {len(records)}; unmatched files: {len(missing)}")
    return output_manifest_path


## 4. Model A: isolated sign recognition

This temporal Conv1d classifier mirrors the current backend's 64×225 feature contract and `IsolatedWordNet` checkpoint shape. Labels remain local to the selected dataset. Validation data selects the best checkpoint; test metrics are calculated only after model selection.

In [ ]:
class IsolatedWordNet(nn.Module):
    def __init__(self, class_count):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Conv1d(225, 64, kernel_size=5, padding=2),
            nn.GELU(),
            nn.Conv1d(64, 96, kernel_size=3, padding=1),
            nn.GELU(),
            nn.AdaptiveAvgPool1d(1),
            nn.Flatten(),
            nn.Linear(96, class_count),
        )

    def forward(self, features):
        return self.layers(features.transpose(1, 2))


def classification_metrics(y_true, y_pred, class_count, train_classes=None):
    matrix = np.zeros((class_count, class_count), dtype=np.int64)
    for truth, pred in zip(y_true, y_pred):
        matrix[int(truth), int(pred)] += 1
    precision, recall, f1 = [], [], []
    for i in range(class_count):
        tp = int(matrix[i, i])
        pden, rden = int(matrix[:, i].sum()), int(matrix[i, :].sum())
        p = tp / pden if pden else 0.0
        r = tp / rden if rden else 0.0
        precision.append(p)
        recall.append(r)
        f1.append(2 * p * r / (p + r) if p + r else 0.0)
    accuracy = float(np.trace(matrix) / max(matrix.sum(), 1))

    # Evaluate seen vs. unseen classes if training split classes were provided
    unseen_in_train = []
    seen_f1 = []
    if train_classes is not None:
        train_set = set(train_classes)
        unseen_in_train = sorted(set(range(class_count)) - train_set)
        seen_f1 = [f1[i] for i in train_set if i < class_count]

    return {
        "accuracy": accuracy,
        "macro_precision": float(np.mean(precision)),
        "macro_recall": float(np.mean(recall)),
        "macro_f1": float(np.mean(f1)),
        "seen_classes_macro_f1": float(np.mean(seen_f1)) if seen_f1 else float(np.mean(f1)),
        "unseen_classes_in_training": unseen_in_train,
        "per_class_recall": {str(i): recall[i] for i in range(class_count)},
        "confusion_matrix": matrix.tolist(),
    }


def atomic_torch_save(payload, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + ".tmp")
    torch.save(payload, temporary)
    os.replace(temporary, path)


def atomic_json_save(payload, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + ".tmp")
    temporary.write_text(json.dumps(payload, indent=2, ensure_ascii=False), encoding="utf-8")
    os.replace(temporary, path)


def _tuple_tree(value):
    return tuple(_tuple_tree(item) for item in value) if isinstance(value, list) else value


def capture_rng_state():
    name, keys, position, has_gauss, cached = np.random.get_state()
    return {
        "python_rng_json": json.dumps(random.getstate()),
        "numpy_rng": {"name": name, "keys": keys.tolist(), "position": position,
                      "has_gauss": has_gauss, "cached_gaussian": cached},
        "torch_rng": torch.get_rng_state(),
        "cuda_rng": torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None,
    }


def restore_rng_state(state):
    random.setstate(_tuple_tree(json.loads(state["python_rng_json"])))
    numpy_rng = state["numpy_rng"]
    np.random.set_state((numpy_rng["name"], np.asarray(numpy_rng["keys"], dtype=np.uint32),
                         numpy_rng["position"], numpy_rng["has_gauss"], numpy_rng["cached_gaussian"]))
    torch.set_rng_state(state["torch_rng"])
    if torch.cuda.is_available() and state.get("cuda_rng") is not None:
        torch.cuda.set_rng_state_all(state["cuda_rng"])


def _classification_eval(model, loader, class_count, train_classes=None):
    model.eval()
    truths, predictions, total_loss, total = [], [], 0.0, 0
    with torch.inference_mode():
        for features, labels in loader:
            logits = model(features.to(DEVICE))
            total_loss += float(nn.functional.cross_entropy(logits, labels.to(DEVICE), reduction="sum").item())
            total += len(labels)
            truths.extend(labels.tolist())
            predictions.extend(logits.argmax(dim=1).cpu().tolist())
    metrics = classification_metrics(truths, predictions, class_count, train_classes)
    metrics["loss"] = total_loss / max(total, 1)
    return metrics


def _classification_examples(model, rows, labels, limit=8):
    examples = []
    model.eval()
    with torch.inference_mode():
        for row in rows[:limit]:
            logits = model(torch.from_numpy(row["features"][None]).to(DEVICE))[0]
            probabilities = torch.softmax(logits, dim=-1)
            confidence, index = probabilities.max(dim=0)
            examples.append({
                "path": row["path"], "ground_truth": row["label"],
                "prediction": labels[int(index)], "confidence": float(confidence),
                "accepted_at_backend_threshold": float(confidence) >= 0.5,
            })
    return examples


def train_recognition():
    if not CONFIRM_DATASET_RIGHTS:
        raise PermissionError("Confirm attached dataset rights in configuration before training.")
    if ISOLATED_LICENSE.strip().lower() in ("", "unverified", "unknown", "none"):
        raise PermissionError("Set ISOLATED_LICENSE after verifying source terms; unresolved-license data is disabled.")
    records = load_isolated_records()
    signer_complete = bool(ISOLATED_SIGNER_COLUMN) and all(r["signer"] for r in records)
    group_key = "signer" if signer_complete else "video_id"
    if not signer_complete:
        print("Signer metadata absent/incomplete: split is video-independent only; signer generalization is unverified.")
    split = split_grouped(records, group_key)
    video_sets = {name: {r["video_id"] for r in rows} for name, rows in split.items()}
    if (video_sets["train"] & video_sets["validation"] or
        video_sets["train"] & video_sets["test"] or
        video_sets["validation"] & video_sets["test"]):
        raise ValueError("Video leakage detected in recognition split.")
    split = {name: limit_split(rows, MAX_SAMPLES_PER_SPLIT, SEED + i) for i, (name, rows) in enumerate(split.items())}
    labels = sorted({r["label"] for r in records})
    label_to_id = {label: i for i, label in enumerate(labels)}

    train_class_indices = {label_to_id[r["label"]] for r in split["train"]}
    absent_from_train = sorted(set(range(len(labels))) - train_class_indices)

    # Preflight Summary
    print("=" * 70)
    print("MODEL A: ISOLATED SIGN RECOGNITION PREFLIGHT SUMMARY")
    print("=" * 70)
    print(f"Total valid samples: {len(records):,}")
    print(f"Total vocabulary classes: {len(labels)}")
    print(f"Samples per split: {', '.join(f'{k}={len(v)}' for k, v in split.items())}")
    print(f"Class coverage by split:")
    for name, rows in split.items():
        present = Counter(r['label'] for r in rows)
        absent = sorted(set(labels) - set(present))
        print(f"  {name}: {len(present)}/{len(labels)} classes present ({len(rows)} samples)")
        if absent:
            print(f"    WARNING: {len(absent)} classes absent from {name} split: {absent[:6]}{'...' if len(absent) > 6 else ''}")
    if absent_from_train:
        print(f"CRITICAL: {len(absent_from_train)} classes are absent from the training split. Evaluation will track seen vs. unseen classes.")
    print(f"Signer/video overlap:")
    print(f"  train & validation = {len(video_sets['train'] & video_sets['validation'])}")
    print(f"  train & test = {len(video_sets['train'] & video_sets['test'])}")
    print(f"  validation & test = {len(video_sets['validation'] & video_sets['test'])}")
    print(f"Feature dimensions: {records[0]['features'].shape} (64 frames x 225 coordinates)")
    print(f"Dataset name: {ISOLATED_DATASET_NAME}; License: {ISOLATED_LICENSE}")
    print(f"Training mode: {TRAINING_MODE}; Scope: {TRAINING_SCOPE}")
    print("=" * 70)

    def loader(rows, shuffle):
        x = torch.from_numpy(np.stack([r["features"] for r in rows]))
        y = torch.tensor([label_to_id[r["label"]] for r in rows], dtype=torch.long)
        return DataLoader(TensorDataset(x, y), batch_size=BATCH_SIZE, shuffle=shuffle)
    train_loader = loader(split["train"], True)
    val_loader = loader(split["validation"], False)
    train_eval_loader = loader(split["train"], False)
    output = OUTPUT_ROOT / "recognition"
    output.mkdir(parents=True, exist_ok=True)
    state_path = output / "training_state.pt"
    model_path = output / "model.pt"
    if RESUME_RECOGNITION:
        if not state_path.is_file():
            raise FileNotFoundError(f"Resume requested but no recovery checkpoint exists: {state_path}")
    elif state_path.exists() or model_path.exists():
        raise FileExistsError(f"Output exists at {output}; set RESUME_RECOGNITION=True to resume, or choose another output.")
    model = IsolatedWordNet(len(labels)).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)
    history, best_f1, start_epoch = [], -1.0, 0
    data_fingerprint = fingerprint_rows(records, ("path", "label", "video_id", "signer"))
    training_config = {"batch_size": BATCH_SIZE, "learning_rate": LEARNING_RATE,
                       "seed": SEED, "max_samples_per_split": MAX_SAMPLES_PER_SPLIT,
                       "training_mode": TRAINING_MODE, "training_scope": TRAINING_SCOPE}
    if RESUME_RECOGNITION:
        state = torch.load(state_path, map_location=DEVICE, weights_only=True)
        if (state["labels"] != labels or state["group_key"] != group_key
                or state["data_fingerprint"] != data_fingerprint
                or state["training_config"] != training_config):
            raise ValueError("Recognition checkpoint data contract differs from current manifest.")
        model.load_state_dict(state["state_dict"])
        optimizer.load_state_dict(state["optimizer"])
        history, best_f1, start_epoch = state["history"], state["best_f1"], state["epoch"] + 1
        restore_rng_state(state)
    if EPOCHS <= 0:
        raise ValueError("EPOCHS must be positive.")
    for epoch in range(start_epoch, EPOCHS):
        model.train()
        total_loss, total, correct = 0.0, 0, 0
        for features, targets in train_loader:
            features, targets = features.to(DEVICE), targets.to(DEVICE)
            optimizer.zero_grad(set_to_none=True)
            logits = model(features)
            loss = nn.functional.cross_entropy(logits, targets)
            loss.backward()
            optimizer.step()
            total_loss += float(loss.item()) * len(targets)
            total += len(targets)
            correct += int((logits.argmax(1) == targets).sum())
        val = _classification_eval(model, val_loader, len(labels), train_class_indices)
        row = {"epoch": epoch + 1, "train_loss": total_loss / max(total, 1),
               "train_accuracy": correct / max(total, 1), "validation_accuracy": val["accuracy"],
               "validation_loss": val["loss"], "validation_macro_f1": val["macro_f1"],
               "validation_seen_classes_macro_f1": val["seen_classes_macro_f1"]}
        history.append(row)
        if val["macro_f1"] > best_f1:
            best_f1 = val["macro_f1"]
            atomic_torch_save({"state_dict": model.state_dict()}, model_path)
        atomic_torch_save({
            "state_dict": model.state_dict(), "optimizer": optimizer.state_dict(),
            "epoch": epoch, "best_f1": best_f1, "history": history, "labels": labels,
            "group_key": group_key, "data_fingerprint": data_fingerprint,
            "training_config": training_config, **capture_rng_state(),
        }, state_path)
        print(row)
    model.load_state_dict(torch.load(model_path, map_location=DEVICE, weights_only=True)["state_dict"])
    test_loader = loader(split["test"], False)
    test_metrics = _classification_eval(model, test_loader, len(labels), train_class_indices)
    report = {
        "status": "EXPERIMENTAL_NOT_PROMOTED",
        "scope": TRAINING_SCOPE,
        "training_mode": TRAINING_MODE,
        "dataset": ISOLATED_DATASET_NAME, "license": ISOLATED_LICENSE,
        "signer_independent": signer_complete, "split_group": group_key,
        "rows": {name: len(rows) for name, rows in split.items()},
        "training": _classification_eval(model, train_eval_loader, len(labels), train_class_indices),
        "validation": _classification_eval(model, val_loader, len(labels), train_class_indices),
        "test": test_metrics,
        "representative_predictions": _classification_examples(
            model, split["test"], labels
        ),
        "history": history,
        "classes_absent_from_training": [labels[i] for i in absent_from_train],
    }
    atomic_json_save(report, output / "evaluation_results.json")
    atomic_json_save(labels, output / "labels.json")
    model_config = {"architecture": "IsolatedWordNet", "class_count": len(labels),
                    "input_shape": [64, 225], "feature_version": "pose_hands_root_scale_v1",
                    "training_scope": TRAINING_SCOPE, "seed": SEED}
    preprocess = {"source_shape": [None, 543, 3], "selected_landmarks": "pose 0:33 + hands 501:543",
                  "frames": 64, "features_per_frame": 225, "normalization": "hip-center / median shoulder width"}
    digest = hashlib.sha256(model_path.read_bytes()).hexdigest()
    metadata = {
        "format": "signsage_isolated_words_v1", "labels": labels,
        "feature_version": "pose_hands_root_scale_v1", "feature_dim": 225,
        "confidence_threshold": 0.5, "checkpoint_sha256": digest,
        "evaluation": report, "promotion_eligible": False,
        "dataset": ISOLATED_DATASET_NAME, "license": ISOLATED_LICENSE,
        "raw_videos_included": False,
    }
    atomic_json_save(model_config, output / "model_config.json")
    atomic_json_save(preprocess, output / "preprocessing_config.json")
    atomic_json_save(metadata, output / "metadata.json")
    atomic_json_save(history, output / "training_history.json")
    print("Recognition held-out test:", report["test"])
    if absent_from_train:
        print(f"Note: test macro_f1 includes {len(absent_from_train)} unseen classes. Seen-classes macro F1: {test_metrics['seen_classes_macro_f1']:.4f}")
    print("Recognition uncertainty: backend returns accepted=false below confidence threshold.")
    return report


if TRAIN_RECOGNITION:
    RECOGNITION_REPORT = train_recognition()
else:
    print("Model A training disabled. Attach a validated manifest and explicitly enable TRAIN_RECOGNITION.")


## 5. Model B: aligned pose-to-English data

GloSL uses its published `train/dev/test` split only after verifying `video_ref` disjointness. iSign and canonical manifests split by source video; complete signer IDs take precedence. `MAX_SAMPLES_PER_SPLIT` bounds Kaggle work and keeps the selected subset deterministic.

In [ ]:
def _pose_path_index(root):
    root = Path(root)
    if not root.is_dir():
        raise FileNotFoundError(f"Pose root not found: {root}")
    index = {}
    for path in root.rglob("*"):
        if path.is_file() and path.suffix.lower() in (".pose", ".npy", ".npz"):
            index.setdefault(path.stem, path)
    return index


def _resolve_glosl_pose(value, root):
    root = Path(root)
    val_str = str(value).replace("\\", "/")
    # Direct path under root
    direct = root / val_str
    if direct.is_file():
        return direct
    # Under poses/ subdirectory
    under_poses = root / "poses" / val_str
    if under_poses.is_file():
        return under_poses
    # Stripped poses/ prefix if already present in value
    if val_str.startswith("poses/"):
        stripped = root / val_str[6:]
        if stripped.is_file():
            return stripped
    # Actionable failure
    subdirs = [p.name for p in root.iterdir() if p.is_dir()] if root.exists() else []
    raise FileNotFoundError(
        f"GloSL pose file not found: {value}\n"
        f"Attempted: {direct} and {under_poses}\n"
        f"TRANSLATION_POSE_ROOT={root.resolve()} (subdirs: {subdirs})\n"
        f"Ensure TRANSLATION_POSE_ROOT points to the root of the extracted GloSL dataset containing 'poses/'."
    )


def load_translation_records():
    if TRANSLATION_MANIFEST is None:
        raise ValueError("Set TRANSLATION_MANIFEST to attached annotations before enabling translation.")
    rows, fields = read_csv_rows(TRANSLATION_MANIFEST)
    root = Path(TRANSLATION_POSE_ROOT)
    show_input_tree(root, limit=10)
    fmt = TRANSLATION_FORMAT
    if fmt == "auto":
        if {"pose_file", "text"}.issubset(fields):
            fmt = "glosl"
        elif {"uid", "text"}.issubset(fields):
            fmt = "isign"
        elif {"pose_path", "text"}.issubset(fields):
            fmt = "canonical"
        else:
            raise ValueError(f"Cannot infer translation format from fields {fields}; set TRANSLATION_FORMAT.")
    pose_index = None
    records = []
    for number, row in enumerate(rows, start=2):
        text = str(row.get("text", "")).strip()
        if not text:
            raise ValueError(f"Translation row {number} has empty English target.")
        if fmt == "glosl":
            value = row.get("pose_file", "")
            path = _resolve_glosl_pose(value, root)
            video = str(row.get("video_ref", "")).strip()
            split_name = str(row.get("split", "")).strip().lower()
            signer = str(row.get("signer", "")).strip()
        elif fmt == "isign":
            uid = str(row.get("uid", "")).strip()
            if not uid:
                raise ValueError(f"Translation row {number} has empty uid.")
            if pose_index is None:
                pose_index = _pose_path_index(root)
            path = pose_index.get(uid)
            if path is None:
                raise FileNotFoundError(f"No attached pose file matches iSign uid {uid!r} in {root}.")
            # iSign UIDs are sample identifiers. Use explicit video_id/signer metadata if provided;
            # otherwise treat each sample independently to avoid false assumptions.
            video = str(row.get("video_id", "")).strip() or uid
            signer = str(row.get("signer", "")).strip()
            split_name = str(row.get("split", "")).strip().lower()
        elif fmt == "canonical":
            path = resolve_input_path(row.get("pose_path", ""), root)
            video = str(row.get("video_id", "")).strip()
            split_name = str(row.get("split", "")).strip().lower()
            signer = str(row.get("signer", "")).strip()
        else:
            raise ValueError(f"Unsupported TRANSLATION_FORMAT={fmt!r}")
        if not video:
            raise ValueError(f"Translation row {number} lacks source video identity.")
        if not path.is_file():
            raise FileNotFoundError(f"Translation row {number} pose file missing: {path}")
        split_name = {"dev": "validation", "val": "validation"}.get(split_name, split_name)
        records.append({"path": path, "text": text, "video_id": video, "signer": signer,
                        "official_split": split_name, "uid": str(row.get("uid") or row.get("id") or number)})
    print(f"translation format={fmt}; aligned rows={len(records):,}; annotation fields={fields}")
    return records, fmt


def assign_translation_splits(records):
    signer_complete = all(row["signer"] for row in records)
    if signer_complete:
        split = split_grouped(records, "signer")
        policy = "signer-independent"
    elif all(row["official_split"] in ("train", "validation", "test") for row in records):
        split = {name: [r for r in records if r["official_split"] == name]
                 for name in ("train", "validation", "test")}
        policy = "published source-video split; signer independence unavailable"
    else:
        split = split_grouped(records, "video_id")
        policy = "source-video-independent; signer independence unavailable"
    video_sets = {name: {r["video_id"] for r in rows} for name, rows in split.items()}
    pairs = [("train", "validation"), ("train", "test"), ("validation", "test")]
    for left, right in pairs:
        overlap = video_sets[left] & video_sets[right]
        if overlap:
            raise ValueError(f"Video leakage between {left} and {right}: {len(overlap)} shared videos.")
    if signer_complete:
        signer_sets = {name: {r["signer"] for r in rows} for name, rows in split.items()}
        for left, right in pairs:
            if signer_sets[left] & signer_sets[right]:
                raise ValueError(f"Signer leakage between {left} and {right}.")
    print(f"split policy: {policy}; " + "; ".join(f"{k}={len(v):,}" for k, v in split.items()))
    return split, signer_complete, policy


def load_translation_split(split_rows):
    loaded, errors = [], []
    for row in split_rows:
        try:
            landmarks = load_landmarks(row["path"])
            norm = normalize_pose_sequence(landmarks.reshape(len(landmarks), RAW_FEATURE_DIM))
            frames, length = resize_sequence(norm, MAX_FRAMES)
            loaded.append({**row, "frames": frames, "length": length})
        except Exception as exc:
            errors.append(f"{row['uid']} ({row['path']}): {exc}")
    if errors:
        print(f"Invalid aligned pose samples: {len(errors)}")
        print("\n".join(errors[:20]))
        raise ValueError("Translation validation failed; invalid samples were not discarded.")
    if not loaded:
        raise ValueError("No valid pose-to-English pairs in split.")
    return loaded


def prepare_translation_data():
    if not CONFIRM_DATASET_RIGHTS:
        raise PermissionError("Confirm attached dataset rights in configuration before training.")
    records, fmt = load_translation_records()
    dataset_name = TRANSLATION_DATASET_NAME
    dataset_license = TRANSLATION_LICENSE
    if fmt == "glosl":
        dataset_name, dataset_license = "GloSL-ISL", "CC BY-NC 4.0"
    elif fmt == "isign":
        dataset_name, dataset_license = "iSign", "CC BY-NC-SA 4.0"
    if "nc" in dataset_license.lower() and not ALLOW_RESEARCH_ONLY_DATASETS:
        raise PermissionError(f"{dataset_name} is non-commercial/research-only. Set ALLOW_RESEARCH_ONLY_DATASETS=True only for research.")
    splits, signer_complete, policy = assign_translation_splits(records)
    splits = {name: limit_split(rows, MAX_SAMPLES_PER_SPLIT, SEED + i)
              for i, (name, rows) in enumerate(splits.items())}
    loaded = {name: load_translation_split(rows) for name, rows in splits.items()}

    video_sets = {name: {r["video_id"] for r in rows} for name, rows in loaded.items()}
    unique_targets = len({row["text"] for name in loaded for row in loaded[name]})

    # Preflight Summary
    print("=" * 70)
    print("MODEL B: CONTINUOUS ISL-TO-ENGLISH TRANSLATION PREFLIGHT SUMMARY")
    print("=" * 70)
    print(f"Total valid aligned pairs: {len(records):,}")
    print(f"Unique English target sentences: {unique_targets:,}")
    print(f"Samples per split: {', '.join(f'{k}={len(v)}' for k, v in loaded.items())}")
    print(f"Signer / video overlap:")
    print(f"  train & validation = {len(video_sets['train'] & video_sets['validation'])}")
    print(f"  train & test = {len(video_sets['train'] & video_sets['test'])}")
    print(f"  validation & test = {len(video_sets['validation'] & video_sets['test'])}")
    print(f"Source format: {fmt}; Split policy: {policy}")
    print(f"Feature dimensions: {loaded['train'][0]['frames'].shape} -> model input {RAW_FEATURE_DIM * 2}")
    print(f"Dataset name: {dataset_name}; License: {dataset_license}")
    print(f"Training mode: {TRAINING_MODE}; Scope: {TRAINING_SCOPE}")
    print("=" * 70)

    return loaded, signer_complete, policy, fmt, len(records), dataset_name, dataset_license


def translation_token_ids(text, tokenizer):
    tokens = [tokenizer.bos_id()] + tokenizer.encode(text, out_type=int) + [tokenizer.eos_id()]
    tokens = tokens[:MAX_TOKENS]
    if tokens[-1] != tokenizer.eos_id():
        tokens[-1] = tokenizer.eos_id()
    return tokens


class TranslationDataset(Dataset):
    def __init__(self, rows, tokenizer):
        self.rows = rows
        self.tokenizer = tokenizer
    def __len__(self):
        return len(self.rows)
    def __getitem__(self, index):
        row = self.rows[index]
        return (torch.from_numpy(row["frames"]), int(row["length"]),
                torch.tensor(translation_token_ids(row["text"], self.tokenizer), dtype=torch.long),
                row["text"], row["uid"])


def translation_collate(batch):
    poses, lengths, token_rows, texts, uids = zip(*batch)
    poses = torch.stack(poses)
    lengths = torch.tensor(lengths, dtype=torch.long)
    max_len = max(len(row) for row in token_rows)
    tokens = torch.full((len(batch), max_len), 0, dtype=torch.long)
    for i, row in enumerate(token_rows):
        tokens[i, :len(row)] = row
    return poses, lengths, tokens[:, :-1], tokens[:, 1:].masked_fill(tokens[:, 1:] == 0, -100), list(texts), list(uids)


### Translation model, loss, and resume

The encoder-decoder uses raw 1629-D frames; its frame projection concatenates frame deltas internally, matching backend input width 3258. Source padding mask uses true sequence lengths. Decoder uses causal and target-padding masks; PAD targets are excluded from cross-entropy. Tokenizer is trained only from training targets.

In [ ]:
class FrameProjection(nn.Module):
    def __init__(self, input_dim, d_model):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 384), nn.LayerNorm(384), nn.GELU(), nn.Dropout(0.10),
            nn.Linear(384, d_model), nn.LayerNorm(d_model),
        )
        self.temporal = nn.Conv1d(d_model, d_model, kernel_size=5, padding=2, groups=d_model)
    def forward(self, x):
        delta = torch.zeros_like(x)
        delta[:, 1:] = x[:, 1:] - x[:, :-1]
        return self.temporal(self.net(torch.cat([x, delta], dim=-1)).transpose(1, 2)).transpose(1, 2)


class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len):
        super().__init__()
        pos = torch.arange(max_len, dtype=torch.float32).unsqueeze(1)
        div = torch.exp(torch.arange(0, d_model, 2, dtype=torch.float32) * (-math.log(10000.0) / d_model))
        pe = torch.zeros(max_len, d_model)
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer("pe", pe.unsqueeze(0))
    def forward(self, x):
        return x + self.pe[:, :x.size(1)]


class SignTransformer(nn.Module):
    def __init__(self, input_dim, vocab_size, max_frames, max_tokens, pad_idx,
                 d_model=128, nhead=4, layers=2, ff=512, dropout=0.15, norm_first=False):
        super().__init__()
        self.pad_idx = pad_idx
        self.input_proj = FrameProjection(input_dim, d_model)
        self.src_pos = PositionalEncoding(d_model, max_frames + 4)
        self.tok_embed = nn.Embedding(vocab_size, d_model, padding_idx=pad_idx)
        self.tgt_pos = PositionalEncoding(d_model, max_tokens + 4)
        enc = nn.TransformerEncoderLayer(d_model, nhead, ff, dropout, batch_first=True, norm_first=norm_first)
        dec = nn.TransformerDecoderLayer(d_model, nhead, ff, dropout, batch_first=True, norm_first=norm_first)
        self.encoder = nn.TransformerEncoder(enc, layers)
        self.decoder = nn.TransformerDecoder(dec, layers)
        self.out = nn.Linear(d_model, vocab_size)
        self.scale = d_model ** 0.5
    def encode(self, frames, src_pad):
        return self.encoder(self.src_pos(self.input_proj(frames)), src_key_padding_mask=src_pad)
    def decode(self, tokens, memory, tgt_mask, src_pad, tgt_pad=None):
        x = self.tgt_pos(self.tok_embed(tokens) * self.scale)
        return self.out(self.decoder(x, memory, tgt_mask=tgt_mask,
                    tgt_key_padding_mask=tgt_pad, memory_key_padding_mask=src_pad))
    def forward(self, frames, src_pad, tokens, tgt_mask):
        return self.decode(tokens, self.encode(frames, src_pad), tgt_mask, src_pad,
                           tgt_pad=tokens.eq(self.pad_idx))


def _tokenizer_fingerprint(model_path, tokenizer):
    digest = hashlib.sha256(Path(model_path).read_bytes())
    digest.update(str(tokenizer.get_piece_size()).encode())
    digest.update(f"{tokenizer.pad_id()},{tokenizer.unk_id()},{tokenizer.bos_id()},{tokenizer.eos_id()}".encode())
    return digest.hexdigest()[:16]


def build_tokenizer(train_rows, directory, resume):
    try:
        import sentencepiece as spm
    except ImportError as exc:
        raise ImportError("Translation requires `pip install sentencepiece`.") from exc
    directory = Path(directory)
    directory.mkdir(parents=True, exist_ok=True)
    model_path = directory / "signsage_en_sp.model"
    metadata_path = directory / "tokenizer_metadata.json"
    corpus_hash = hashlib.sha256("\n".join(row["text"] for row in train_rows).encode()).hexdigest()
    if resume:
        if not model_path.is_file() or not metadata_path.is_file():
            raise FileNotFoundError("Resume requested but tokenizer artifacts are missing.")
        meta = json.loads(metadata_path.read_text(encoding="utf-8"))
        if meta["training_text_sha256"] != corpus_hash:
            raise ValueError("Training text changed since checkpoint; refusing mismatched tokenizer resume.")
    else:
        if model_path.exists():
            raise FileExistsError(f"Tokenizer already exists at {model_path}; resume or use a new output directory.")
        with tempfile.TemporaryDirectory() as temp_dir:
            corpus = Path(temp_dir) / "train.txt"
            corpus.write_text("\n".join(row["text"].replace("\n", " ") for row in train_rows) + "\n", encoding="utf-8")
            spm.SentencePieceTrainer.train(
                input=str(corpus), model_prefix=str(directory / "signsage_en_sp"),
                vocab_size=SP_VOCAB_SIZE, model_type="unigram", character_coverage=0.9995,
                pad_id=0, unk_id=1, bos_id=2, eos_id=3, hard_vocab_limit=False,
                shuffle_input_sentence=False, num_threads=1, minloglevel=2,
            )
        atomic_json_save({"training_text_sha256": corpus_hash}, metadata_path)
    tokenizer = spm.SentencePieceProcessor(model_file=str(model_path))
    actual = (tokenizer.pad_id(), tokenizer.unk_id(), tokenizer.bos_id(), tokenizer.eos_id())
    if actual != (0, 1, 2, 3):
        raise ValueError(f"Unexpected SentencePiece special-token IDs: {actual}")
    return tokenizer, model_path


def source_padding_mask(lengths, max_len):
    return torch.arange(max_len, device=lengths.device).unsqueeze(0) >= lengths.unsqueeze(1)


def causal_mask(size, device):
    return torch.triu(torch.ones(size, size, dtype=torch.bool, device=device), diagonal=1)


def _translation_loss_epoch(model, loader, optimizer=None):
    training = optimizer is not None
    model.train(training)
    total_loss, total_tokens = 0.0, 0
    grad_context = torch.enable_grad if training else torch.no_grad
    with grad_context():
        for poses, lengths, decoder_input, targets, _, _ in loader:
            poses, lengths = poses.to(DEVICE), lengths.to(DEVICE)
            decoder_input, targets = decoder_input.to(DEVICE), targets.to(DEVICE)
            source_mask = source_padding_mask(lengths, poses.size(1))
            logits = model(poses, source_mask, decoder_input, causal_mask(decoder_input.size(1), DEVICE))
            loss = nn.functional.cross_entropy(logits.reshape(-1, logits.size(-1)), targets.reshape(-1), ignore_index=-100)
            if training:
                optimizer.zero_grad(set_to_none=True)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
            valid = int((targets != -100).sum())
            total_loss += float(loss.item()) * valid
            total_tokens += valid
    return total_loss / max(total_tokens, 1)


def _ban_ngram(logits, ids, n=3):
    if n <= 1 or len(ids) < n - 1:
        return logits
    prefix = tuple(ids[-(n - 1):])
    for start in range(len(ids) - n + 1):
        gram = tuple(ids[start:start + n])
        if gram[:-1] == prefix:
            logits[gram[-1]] = -torch.inf
    return logits


@torch.inference_mode()
def greedy_decode_batch(model, poses, lengths, tokenizer, max_new_tokens=MAX_TOKENS - 1):
    model.eval()
    poses, lengths = poses.to(DEVICE), lengths.to(DEVICE)
    source_mask = source_padding_mask(lengths, poses.size(1))
    memory = model.encode(poses, source_mask)
    ids = torch.full((len(poses), 1), tokenizer.bos_id(), dtype=torch.long, device=DEVICE)
    finished = torch.zeros(len(poses), dtype=torch.bool, device=DEVICE)
    for step in range(max_new_tokens):
        logits = model.decode(ids, memory, causal_mask(ids.size(1), DEVICE), source_mask)[:, -1].float()
        logits[:, tokenizer.pad_id()] = -torch.inf
        logits[:, tokenizer.bos_id()] = -torch.inf
        logits[:, tokenizer.unk_id()] = -torch.inf
        if step < 1:
            logits[:, tokenizer.eos_id()] = -torch.inf
        for i in range(len(ids)):
            if finished[i]:
                logits[i].fill_(-torch.inf)
                logits[i, tokenizer.eos_id()] = 0.0
            else:
                _ban_ngram(logits[i], ids[i].tolist())
                if not torch.isfinite(logits[i]).any():
                    logits[i, tokenizer.eos_id()] = 0.0
        next_ids = logits.argmax(dim=-1)
        ids = torch.cat([ids, next_ids[:, None]], dim=1)
        finished |= next_ids.eq(tokenizer.eos_id())
        if bool(finished.all()):
            break
    blocked = {tokenizer.eos_id(), tokenizer.pad_id(), tokenizer.bos_id(), tokenizer.unk_id()}
    return [tokenizer.decode([int(x) for x in row[1:] if int(x) not in blocked]) for row in ids]


def _edit_distance(left, right):
    previous = list(range(len(right) + 1))
    for i, a in enumerate(left, 1):
        current = [i]
        for j, b in enumerate(right, 1):
            current.append(min(current[-1] + 1, previous[j] + 1, previous[j - 1] + (a != b)))
        previous = current
    return previous[-1]


def _lcs_length(a, b):
    previous = [0] * (len(b) + 1)
    for x in a:
        current = [0]
        for j, y in enumerate(b, 1):
            current.append(previous[j - 1] + 1 if x == y else max(previous[j], current[-1]))
        previous = current
    return previous[-1]


def translation_metrics(references, predictions):
    refs = [str(x).lower().split() for x in references]
    hyps = [str(x).lower().split() for x in predictions]
    edits = sum(_edit_distance(r, h) for r, h in zip(refs, hyps))
    ref_words = sum(len(r) for r in refs)
    clipped, totals = [0] * 4, [0] * 4
    for ref, hyp in zip(refs, hyps):
        for n in range(1, 5):
            ref_counts = Counter(tuple(ref[i:i+n]) for i in range(max(0, len(ref)-n+1)))
            hyp_counts = Counter(tuple(hyp[i:i+n]) for i in range(max(0, len(hyp)-n+1)))
            clipped[n-1] += sum(min(count, ref_counts[gram]) for gram, count in hyp_counts.items())
            totals[n-1] += sum(hyp_counts.values())
    precisions = [(clipped[i] + 1) / (totals[i] + 1) for i in range(4)]
    hyp_words = sum(len(h) for h in hyps)
    brevity = min(1.0, math.exp(1 - ref_words / max(hyp_words, 1)))
    bleu = 100.0 * brevity * math.exp(sum(math.log(p) for p in precisions) / 4)
    rouge = []
    for ref, hyp in zip(refs, hyps):
        common = _lcs_length(ref, hyp)
        p = common / len(hyp) if hyp else 0.0
        r = common / len(ref) if ref else 0.0
        rouge.append(2 * p * r / (p + r) if p + r else 0.0)
    normalized = [" ".join(x) for x in hyps]
    frequency = Counter(normalized)
    diversity = len(frequency) / max(len(normalized), 1)
    return {
        "wer": edits / max(ref_words, 1),
        "bleu4": bleu,
        "rouge_l": float(np.mean(rouge)) if rouge else 0.0,
        "prediction_diversity": diversity,
        "top_prediction_frequency": max(frequency.values(), default=0) / max(len(normalized), 1),
        "collapsed": diversity < 0.05 or (max(frequency.values(), default=0) / max(len(normalized), 1) > 0.5),
        "samples": len(refs),
    }


def evaluate_translation(model, loader, tokenizer):
    references, predictions, examples = [], [], []
    for poses, lengths, _, _, texts, uids in loader:
        decoded = greedy_decode_batch(model, poses, lengths, tokenizer)
        references.extend(texts)
        predictions.extend(decoded)
        for uid, reference, prediction in zip(uids, texts, decoded):
            if len(examples) < 8:
                examples.append({"uid": uid, "reference": reference, "prediction": prediction})
    return translation_metrics(references, predictions), examples


def train_translation():
    loaded, signer_complete, split_policy, fmt, available_rows, dataset_name, dataset_license = prepare_translation_data()
    output = OUTPUT_ROOT / "translation"
    output.mkdir(parents=True, exist_ok=True)
    checkpoint_dir = output / "checkpoint"
    tokenizer_dir = output / "tokenizer"
    state_path = output / "training_state.pt"
    best_path = checkpoint_dir / "signsage_transformer_full.pt"
    if RESUME_TRANSLATION:
        if not state_path.is_file():
            raise FileNotFoundError(f"Resume requested but no recovery checkpoint exists: {state_path}")
    elif state_path.exists() or best_path.exists() or (tokenizer_dir / "signsage_en_sp.model").exists():
        raise FileExistsError(f"Output exists at {output}; resume or choose a fresh output directory.")
    tokenizer, tokenizer_path = build_tokenizer(loaded["train"], tokenizer_dir, RESUME_TRANSLATION)
    loaders = {
        name: DataLoader(TranslationDataset(rows, tokenizer), batch_size=BATCH_SIZE,
                         shuffle=name == "train", num_workers=0, collate_fn=translation_collate)
        for name, rows in loaded.items()
    }
    model_config = {
        "input_dim": RAW_FEATURE_DIM * 2, "raw_feature_dim": RAW_FEATURE_DIM,
        "vocab_size": tokenizer.get_piece_size(), "max_frames": MAX_FRAMES, "max_tokens": MAX_TOKENS,
        "d_model": MODEL_DIM, "nhead": ATTENTION_HEADS, "layers": TRANSFORMER_LAYERS,
        "ff": FEED_FORWARD_DIM, "dropout": 0.15, "norm_first": False,
        "training_scope": TRAINING_SCOPE,
    }
    model = SignTransformer(
        input_dim=model_config["input_dim"], vocab_size=model_config["vocab_size"],
        max_frames=MAX_FRAMES, max_tokens=MAX_TOKENS, pad_idx=tokenizer.pad_id(),
        d_model=MODEL_DIM, nhead=ATTENTION_HEADS, layers=TRANSFORMER_LAYERS,
        ff=FEED_FORWARD_DIM, dropout=0.15, norm_first=False,
    ).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)
    history, best_val, start_epoch = [], float("inf"), 0
    data_fingerprint = fingerprint_rows(
        [row for name in ("train", "validation", "test") for row in loaded[name]],
        ("uid", "text", "path", "video_id", "signer"),
    )
    training_config = {"batch_size": BATCH_SIZE, "learning_rate": LEARNING_RATE,
                       "seed": SEED, "max_samples_per_split": MAX_SAMPLES_PER_SPLIT,
                       "training_mode": TRAINING_MODE, "training_scope": TRAINING_SCOPE}
    if RESUME_TRANSLATION:
        state = torch.load(state_path, map_location=DEVICE, weights_only=True)
        if (state["model_config"] != model_config
                or state["data_fingerprint"] != data_fingerprint
                or state["training_config"] != training_config):
            raise ValueError("Translation data, model, or training config differs from recovery checkpoint.")
        model.load_state_dict(state["state_dict"])
        optimizer.load_state_dict(state["optimizer"])
        history, best_val, start_epoch = state["history"], state["best_val"], state["epoch"] + 1
        restore_rng_state(state)
    if EPOCHS <= 0:
        raise ValueError("EPOCHS must be positive.")
    for epoch in range(start_epoch, EPOCHS):
        train_loss = _translation_loss_epoch(model, loaders["train"], optimizer)
        val_loss = _translation_loss_epoch(model, loaders["validation"])
        row = {"epoch": epoch + 1, "train_loss": train_loss, "validation_loss": val_loss}
        history.append(row)
        if val_loss < best_val:
            best_val = val_loss
            payload = {"state_dict": model.state_dict(), "model_config": model_config,
                       "tokenizer_fingerprint": _tokenizer_fingerprint(tokenizer_path, tokenizer)}
            atomic_torch_save(payload, best_path)
        atomic_torch_save({
            "state_dict": model.state_dict(), "optimizer": optimizer.state_dict(),
            "epoch": epoch, "best_val": best_val, "history": history,
            "model_config": model_config, "data_fingerprint": data_fingerprint,
            "training_config": training_config, **capture_rng_state(),
        }, state_path)
        print(row)
    model.load_state_dict(torch.load(best_path, map_location=DEVICE, weights_only=True)["state_dict"])
    test_metrics, examples = evaluate_translation(model, loaders["test"], tokenizer)
    report = {
        "status": "EXPERIMENTAL_NOT_PROMOTED", "scope": TRAINING_SCOPE,
        "training_mode": TRAINING_MODE,
        "dataset": dataset_name, "license": dataset_license,
        "source_format": fmt, "signer_independent": signer_complete,
        "split_policy": split_policy, "available_rows": available_rows,
        "rows_used": {name: len(rows) for name, rows in loaded.items()},
        "test_metrics": test_metrics, "examples": examples,
        "prediction_diversity_warning": test_metrics["collapsed"],
        "avatar_signing_solved": False,
    }
    atomic_json_save(report, output / "evaluation_report.json")
    atomic_json_save(history, output / "training_history.json")
    atomic_json_save(model_config, output / "model_config.json")
    atomic_json_save({
        "total_landmark_count": TOTAL_LANDMARKS, "coordinates_per_landmark": 3,
        "source_shape": [None, TOTAL_LANDMARKS, 3], "raw_feature_dim": RAW_FEATURE_DIM,
        "component_order": COMPONENT_ORDER, "component_ranges": COMPONENT_RANGES,
        "coordinate_order": ["x", "y", "z"], "normalization_version": "pose_normalize_v1",
        "max_frames": MAX_FRAMES, "final_model_input_dim": RAW_FEATURE_DIM * 2,
        "full_feature_mask": [1] * RAW_FEATURE_DIM,
    }, output / "landmark_contract.json")
    atomic_json_save({
        "feature_dim": RAW_FEATURE_DIM, "model_input_dim": RAW_FEATURE_DIM * 2,
        "max_frames": MAX_FRAMES, "max_tokens": MAX_TOKENS,
        "vocab_size": tokenizer.get_piece_size(),
        "special_token_ids": {"pad": tokenizer.pad_id(), "unk": tokenizer.unk_id(),
                              "bos": tokenizer.bos_id(), "eos": tokenizer.eos_id()},
        "normalization_version": "pose_normalize_v1",
        "tokenizer_model": "tokenizer/signsage_en_sp.model",
        "checkpoint": "checkpoint/signsage_transformer_full.pt",
        "decoding": {"max_decode_tokens": MAX_TOKENS - 1, "min_output_tokens": 1,
                     "no_repeat_ngram_size": 3, "repetition_penalty": 1.0},
    }, output / "sign_to_text_deployment.json")
    print("Translation held-out test:", test_metrics)
    if test_metrics["collapsed"]:
        print("Warning: repetitive/collapsed output detected; do not present as generalization.")
    return report


if TRAIN_TRANSLATION:
    TRANSLATION_REPORT = train_translation()
else:
    print("Model B training disabled. Attach aligned pose/text data and explicitly enable TRAIN_TRANSLATION.")


## 6. Evaluation and export

Recognition reports training/validation/test separately, including macro metrics and confusion matrix. Translation reports teacher-forced loss curves and held-out WER, corpus BLEU-4, ROUGE-L, output diversity, and examples. Test data never selects checkpoints. No signer-independent generalization claim is made when signer IDs are absent.

## 7–8. Backend integration

- Recognition export follows `backend/app/ml/isolated_words.py`: `model.pt`, `labels.json`, `metadata.json`, `evaluation_results.json`, and matching feature config. Load with `IsolatedWordAdapter.load("outputs/recognition")`; call `adapter.transcribe(raw_landmarks_543)`. Existing confidence/motion rejection remains in backend.
- Translation export follows `SignToTextAdapter`: `checkpoint/signsage_transformer_full.pt`, `tokenizer/signsage_en_sp.model`, `landmark_contract.json`, and `sign_to_text_deployment.json`. Load with `SignToTextAdapter.load("outputs/translation")`; call `adapter.transcribe(raw_landmarks_543, min_motion=0.002, min_confidence=0.0)`. Existing acceptance/rejection behavior remains unchanged.
- Both bundles are marked experimental/pilot and not promotion-ready. No backend/frontend files change. Neither model solves avatar signing. Raw videos are not copied to outputs.

In [ ]:
def synthetic_smoke_check():
    torch.manual_seed(SEED)
    # Recognition forward/backward on canonical feature shape.
    recognition = IsolatedWordNet(3).to(DEVICE)
    x = torch.randn(2, 64, 225, device=DEVICE)
    y = torch.tensor([0, 2], device=DEVICE)
    loss_a = nn.functional.cross_entropy(recognition(x), y)
    loss_a.backward()
    assert torch.isfinite(loss_a)
    # Translation forward/backward with source + causal + target padding masks.
    translation = SignTransformer(input_dim=32, vocab_size=12, max_frames=8, max_tokens=6,
                                  pad_idx=0, d_model=32, nhead=4, layers=1, ff=64).to(DEVICE)
    poses = torch.randn(2, 8, 16, device=DEVICE)
    lengths = torch.tensor([8, 5], device=DEVICE)
    src_mask = source_padding_mask(lengths, poses.size(1))
    tokens = torch.tensor([[2, 4, 5], [2, 6, 0]], device=DEVICE)
    logits = translation(poses, src_mask, tokens, causal_mask(tokens.size(1), DEVICE))
    target = torch.tensor([[4, 5, 3], [6, 3, -100]], device=DEVICE)
    loss_b = nn.functional.cross_entropy(logits.reshape(-1, 12), target.reshape(-1), ignore_index=-100)
    loss_b.backward()
    assert logits.shape == (2, 3, 12) and torch.isfinite(loss_b)
    print(f"Synthetic smoke passed: recognition_loss={loss_a.item():.4f}, translation_loss={loss_b.item():.4f}")


if RUN_SYNTHETIC_SMOKE:
    synthetic_smoke_check()

In [ ]:
if TRAIN_RECOGNITION:
    import matplotlib.pyplot as plt
    history = RECOGNITION_REPORT["history"]
    plt.plot([x["epoch"] for x in history], [x["train_loss"] for x in history], label="train loss")
    plt.plot([x["epoch"] for x in history], [x["validation_loss"] for x in history], label="validation loss")
    plt.xlabel("Epoch")
    plt.ylabel("Cross-entropy")
    plt.legend()
    plt.show()
    print("Training accuracy is separate from held-out validation/test metrics.")
    print(json.dumps(RECOGNITION_REPORT, indent=2))

In [ ]:
if TRAIN_TRANSLATION:
    import matplotlib.pyplot as plt
    history = json.loads((OUTPUT_ROOT / "translation" / "training_history.json").read_text(encoding="utf-8"))
    plt.plot([x["epoch"] for x in history], [x["train_loss"] for x in history], label="train loss")
    plt.plot([x["epoch"] for x in history], [x["validation_loss"] for x in history], label="validation loss")
    plt.xlabel("Epoch")
    plt.ylabel("Masked token cross-entropy")
    plt.legend()
    plt.show()
    print(json.dumps(TRANSLATION_REPORT, indent=2))